# CrewAI — Analyst → Strategist crew

CrewAI models a workflow as a **crew**: you define `Agent`s with a role, a goal, and a
backstory, give them `Task`s, and pick a `Process` (sequential or hierarchical) that
decides the call order for you. You describe *who does what*, not *how the calls chain*.

| Framework | Core mental model | How multi-step flow is defined | Orchestration style |
|---|---|---|---|
| LangGraph | Explicit state graph | You wire nodes (functions/LLM calls) and edges yourself | Manual, graph-based — full control, more code |
| **CrewAI** | Role-based crew | You define `Agent`s (role/goal) and `Task`s; a `Process` (sequential/hierarchical) runs them | Opinionated, role-based — less code, less control |
| Microsoft Agent Framework | Unified successor to AutoGen + Semantic Kernel | Agents come from a chat client; you wire them with a Builder (Sequential, Concurrent, Handoff, GroupChat, Magentic) | Both — low-level like LangGraph, or pre-built patterns like CrewAI |



In [ ]:
%pip install -q crewai anthropic

> **Gotcha:** After the install cell finishes for the first time, use
> **Runtime ▸ Restart session** before running the rest of the notebook. Colab can keep
> an old, incompatible copy of a dependency loaded in memory even after a fresh install
> replaces it on disk.


In [ ]:
import os

def _load(key, default=None):
    """Colab Secrets first, then environment variable, then interactive prompt."""
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val:
            return val
    except Exception:
        pass
    val = os.environ.get(key, default)
    if not val:
        from getpass import getpass
        val = getpass(f"{key}: ")
    return val

ANTHROPIC_API_KEY = _load("ANTHROPIC_API_KEY")
ANTHROPIC_MODEL = _load("ANTHROPIC_MODEL", "claude-sonnet-4-5-20250929")
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

print(f"Using model: {ANTHROPIC_MODEL}")


## The scenario

An assistant reads user interviews, support tickets, and
competitor notes, and is supposed to turn that into a feature recommendation — without
mismatched analysis or hallucinated suggestions.

We'll build that as a **two-step pipeline**:

1. **Analyst** — reads the raw signals, extracts 2–3 themes, and cites which signal(s)
   support each one (a small guardrail against fabrication).
2. **Strategist** — takes *only* the Analyst's themes and recommends ONE feature with a
   one-line rationale. It's told not to invent anything the Analyst didn't surface.

Same signals, same two-step shape, three different frameworks — that's the point of
building all three tonight.

In CrewAI terms: two **Agents** (Analyst, Strategist) each given a **Task**, run as a **sequential Process** — the second Task's `context` is the first Task's output.


In [ ]:
SIGNALS = """
- User interview (Maria, SMB owner): "I got lost during setup and gave up before finishing."
- Support ticket #4821: "Can't find where to invite my team members. Confusing menu."
- Competitor teardown (RivalApp): just shipped a guided setup checklist with a progress bar.
- Support ticket #4903: "Wish there was a quick tour when I first logged in."
""".strip()

print(SIGNALS)


In [ ]:
from crewai import Agent, LLM

# Pinned to the classroom model on purpose: CrewAI's tool-calling / final-answer
# handling has a known compatibility issue with some newer Claude releases (400
# error). max_iter gives the agent room to retry within a turn.
llm = LLM(model=f"anthropic/{ANTHROPIC_MODEL}", max_tokens=400)

analyst = Agent(
    role="Product Signal Analyst",
    goal="Extract 2-3 themes from raw customer signals, citing sources",
    backstory="You read user interviews, tickets, and competitor notes for a living.",
    llm=llm,
    max_iter=15,
    verbose=False,
)

strategist = Agent(
    role="Product Strategist",
    goal="Recommend ONE feature with a one-line rationale, grounded only in the themes",
    backstory="You turn the analyst's themes into a single prioritized call.",
    llm=llm,
    max_iter=15,
    verbose=False,
)


In [ ]:
from crewai import Task, Crew, Process

analyze_task = Task(
    description="Signals:\n{signals}\n\nExtract 2-3 themes as bullet points, citing which signal(s) support each.",
    expected_output="A short bullet list of themes with citations.",
    agent=analyst,
)

recommend_task = Task(
    description="Using ONLY the themes above, recommend ONE feature to prioritize next sprint with a one-line rationale.",
    expected_output="One feature recommendation with a one-line rationale.",
    agent=strategist,
    context=[analyze_task],
)

crew = Crew(
    agents=[analyst, strategist],
    tasks=[analyze_task, recommend_task],
    process=Process.sequential,
    verbose=False,
)

result = await crew.kickoff_async(inputs={"signals": SIGNALS})
print(result)


## How CrewAI differs from LangGraph and Microsoft Agent Framework

| Framework | Core mental model | How multi-step flow is defined | Orchestration style |
|---|---|---|---|
| LangGraph | Explicit state graph | You wire nodes (functions/LLM calls) and edges yourself | Manual, graph-based — full control, more code |
| **CrewAI** | Role-based crew | You define `Agent`s (role/goal) and `Task`s; a `Process` (sequential/hierarchical) runs them | Opinionated, role-based — less code, less control |
| Microsoft Agent Framework | Unified successor to AutoGen + Semantic Kernel | Agents come from a chat client; you wire them with a Builder (Sequential, Concurrent, Handoff, GroupChat, Magentic) | Both — low-level like LangGraph, or pre-built patterns like CrewAI |


**Networking analogy:** CrewAI reads like a NOC escalation runbook. You don't hand-cable
individual links the way LangGraph does — you write job descriptions (Tier-1 Analyst,
Tier-2 Strategist) and pick a process (sequential hand-off), and the framework figures
out the call sequence for you. That's less control, but far less code for a
straightforward hand-off like this one.

Microsoft Agent Framework's `GroupChatBuilder`/`MagenticBuilder` patterns aim at similar
"describe the team, not the wiring" territory, but layered on enterprise features
(sessions, middleware, OpenTelemetry) CrewAI doesn't focus on.

**Try it yourself:** switch `process=Process.sequential` to `Process.hierarchical` and
add a manager `Agent` — CrewAI will now decide the call order itself instead of running
your task list top to bottom.
